# Round 3 — Policy-specific feature effects

**One bounded representation experiment.** Reuse Round 1 and Round 2 controls. Keep the classifier fixed. No neural inference, model download, external data, or automatic submission. This repeatedly inspected 881-comment cohort is exploratory, not an independent holdout or a Kaggle score.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/policy_features.json").is_file())
from scripts.run_policy_features import verify_prior, bounded_compute, figures, write_dashboard
config = json.loads((ROOT / "configs/policy_features.json").read_text())
print("Project:", ROOT)
print("Registered primary:", config["primary"])
print("New CPU fits:", config["new_fits"], "| Saved control fits reused:", config["cached_control_fits"])

Project: /home/sagemaker-user/projects/jigsaw-rule-classifier
Registered primary: condition_lexical
New CPU fits: 12 | Saved control fits reused: 6


## 1. What the previous evidence supports

Round 2's actor-role gain was +0.00624 macro AUC, uncertain and concentrated in legal advice. Removing legal-topic cues erased much of the preceding behavior gain. This motivates testing whether associations should differ by rule, rather than adding another long regex list. The actor model below is an exploratory reference, not a promoted accepted model.

In [2]:
prior, old, previous, directory = verify_prior(ROOT, config)
display(pd.DataFrame(previous["metrics"]).query("variant in ['lexical_control','add_behavior','add_act_roles','behavior_without_topic']"))
print("Round 2 decision:", previous["decision"])
print("Prior checkpoints verified; none refitted.")

,fold,policy,variant,auc,brier,log_loss,queries
0,0,"No Advertising: Spam, referral links, unsolici...",lexical_control,0.673022,0.233599,0.667527,234
1,0,"No Advertising: Spam, referral links, unsolici...",add_behavior,0.675784,0.231577,0.663719,234
2,0,"No Advertising: Spam, referral links, unsolici...",add_act_roles,0.675784,0.231472,0.663380,234
9,0,"No Advertising: Spam, referral links, unsolici...",behavior_without_topic,0.675485,0.231243,0.661525,234
11,1,No legal advice: Do not offer or request legal...,lexical_control,0.641993,0.232466,0.656063,647
12,1,No legal advice: Do not offer or request legal...,add_behavior,0.683940,0.231997,0.710278,647
13,1,No legal advice: Do not offer or request legal...,add_act_roles,0.696415,0.230032,0.716591,647
20,1,No legal advice: Do not offer or request legal...,behavior_without_topic,0.638706,0.240276,0.707843,647


Round 2 decision: DO_NOT_PROMOTE_PRIMARY
Prior checkpoints verified; none refitted.


## 2. Feature construction and honest comparator

Shared + rule-specific lexical and dense feature blocks let one cue have different fitted effects under different rules. Unknown policies receive a zero additional block and retain the shared features. Each conditioned candidate has an equally sized, equal-row-norm shared-copy control, because duplicating features changes effective regularization even at fixed C.

Two families, six configurations, 12 new fits. All six old reference predictions must match reconstructed designs before fitting. Query labels are only joined later for scoring. The target rule is known through supplied support labels; this is not zero-shot evaluation.

In [3]:
result = bounded_compute(ROOT)
print("Decision:", result["decision"])
print("Fits newly completed:", result["new_fits"])
print("Prior control fits reused:", result["reused_prior_controls"])
print("Reference design parity:", result["control_design_parity"])
CHARTS = figures(result)

{"timestamp": "2026-09-11T23:32:14+00:00", "stage": "policy_round3", "event": "started", "elapsed_seconds": 0.0, "stage_elapsed_seconds": 0.0, "total_elapsed_seconds": 0.0}


{"timestamp": "2026-09-11T23:32:21+00:00", "stage": "policy_round3", "event": "all_control_designs_verified", "elapsed_seconds": 6.923, "stage_elapsed_seconds": 6.923, "total_elapsed_seconds": 6.923, "cached_controls": 6, "new_fits": 0}


{"timestamp": "2026-09-11T23:32:22+00:00", "stage": "policy_round3", "event": "candidate_complete", "elapsed_seconds": 7.955, "stage_elapsed_seconds": 7.955, "total_elapsed_seconds": 7.955, "completed": 1, "total": 12, "fold": 0, "variant": "copy_lexical", "new_fits": 1, "reused_fits": 0}


{"timestamp": "2026-09-11T23:32:23+00:00", "stage": "policy_round3", "event": "candidate_complete", "elapsed_seconds": 8.964, "stage_elapsed_seconds": 8.964, "total_elapsed_seconds": 8.964, "completed": 2, "total": 12, "fold": 0, "variant": "condition_lexical", "new_fits": 2, "reused_fits": 0}


{"timestamp": "2026-09-11T23:32:24+00:00", "stage": "policy_round3", "event": "candidate_complete", "elapsed_seconds": 9.692, "stage_elapsed_seconds": 9.692, "total_elapsed_seconds": 9.692, "completed": 3, "total": 12, "fold": 0, "variant": "copy_behavior", "new_fits": 3, "reused_fits": 0}


...

{"timestamp": "2026-09-11T23:32:29+00:00", "stage": "policy_round3", "event": "candidate_complete", "elapsed_seconds": 14.824, "stage_elapsed_seconds": 14.824, "total_elapsed_seconds": 14.824, "completed": 10, "total": 12, "fold": 1, "variant": "condition_behavior", "new_fits": 10, "reused_fits": 0}
{"timestamp": "2026-09-11T23:32:29+00:00", "stage": "policy_round3", "event": "heartbeat", "elapsed_seconds": 15.001, "stage_elapsed_seconds": 15.001, "total_elapsed_seconds": 15.001}


{"timestamp": "2026-09-11T23:32:30+00:00", "stage": "policy_round3", "event": "candidate_complete", "elapsed_seconds": 15.856, "stage_elapsed_seconds": 15.856, "total_elapsed_seconds": 15.856, "completed": 11, "total": 12, "fold": 1, "variant": "copy_both", "new_fits": 11, "reused_fits": 0}


{"timestamp": "2026-09-11T23:32:31+00:00", "stage": "policy_round3", "event": "candidate_complete", "elapsed_seconds": 16.756, "stage_elapsed_seconds": 16.756, "total_elapsed_seconds": 16.756, "completed": 12, "total": 12, "fold": 1, "variant": "condition_both", "new_fits": 12, "reused_fits": 0}


{"timestamp": "2026-09-11T23:32:31+00:00", "stage": "policy_round3", "event": "results_saved", "elapsed_seconds": 17.083, "stage_elapsed_seconds": 17.083, "total_elapsed_seconds": 17.083, "decision": "DO_NOT_PROMOTE_PRIMARY", "new_fits": 12}
{"timestamp": "2026-09-11T23:32:31+00:00", "stage": "policy_round3", "event": "completed", "elapsed_seconds": 17.084, "stage_elapsed_seconds": 17.084, "total_elapsed_seconds": 17.084, "error_type": null}
RESULT: POLICY_ROUND_COMPLETE DECISION: DO_NOT_PROMOTE_PRIMARY


Decision: DO_NOT_PROMOTE_PRIMARY
Fits newly completed: 12
Prior control fits reused: 6
Reference design parity: True


## 3. Per-policy performance and paired uncertainty

Do both policies benefit, or does the mean hide a regression? Intervals condition on fixed predictions and cover only this round's 13 comparisons. They do not account for the full adaptive research history.

In [4]:
display(pd.DataFrame(result["metrics"]))
CHARTS[0].show(renderer="plotly_mimetype")
CHARTS[1].show(renderer="plotly_mimetype")

,fold,policy,variant,auc,brier,log_loss,queries
0,0,"No Advertising: Spam, referral links, unsolici...",lexical_control,0.673022,0.233599,0.667527,234
1,0,"No Advertising: Spam, referral links, unsolici...",add_behavior,0.675784,0.231577,0.663719,234
2,0,"No Advertising: Spam, referral links, unsolici...",add_act_roles,0.675784,0.231472,0.663380,234
3,0,"No Advertising: Spam, referral links, unsolici...",copy_lexical,0.673097,0.237143,0.684000,234
4,0,"No Advertising: Spam, referral links, unsolici...",condition_lexical,0.686604,0.231335,0.669720,234
5,0,"No Advertising: Spam, referral links, unsolici...",copy_behavior,0.675112,0.231728,0.664052,234
6,0,"No Advertising: Spam, referral links, unsolici...",condition_behavior,0.673470,0.243309,0.797146,234
7,0,"No Advertising: Spam, referral links, unsolici...",copy_both,0.672575,0.237451,0.684812,234
8,0,"No Advertising: Spam, referral links, unsolici...",condition_both,0.684142,0.242167,0.797640,234
9,1,No legal advice: Do not offer or request legal...,lexical_control,0.641993,0.232466,0.656063,647


## 4. Policy effect versus scaling effect

The primary must beat both the actor anchor and the shared lexical-copy control. A gain against only the original anchor is insufficient. Zero-padded control columns equalize allocated width; they are not presented as meaningful retained features. Sparse vocabulary is fitted only on eligible training.

In [5]:
display(pd.DataFrame(result["comparisons"]).query("comparison.str.startswith('conditioning:')", engine="python"))
CHARTS[2].show(renderer="plotly_mimetype")
CHARTS[3].show(renderer="plotly_mimetype")

,comparison,candidate,reference,delta_auc,simultaneous_low,simultaneous_high,valid_draws
6,conditioning: lexical,condition_lexical,copy_lexical,0.009704,-0.014177,0.033584,500
7,conditioning: behavior,condition_behavior,copy_behavior,-0.001447,-0.025328,0.022433,500
8,conditioning: both,condition_both,copy_both,0.009340,-0.014540,0.033221,500


## 5. Availability of support labels and fitted effects

The rule-specific block needs legal, supplied training evidence for that rule. The chart reports eligible training counts, not query outcomes. Dense coefficient plots are descriptive associations; they are not substitutes for matched additions/removals. Raw token vocabularies and comment text are not exported.

In [6]:
display(pd.DataFrame(result["support_coverage"]))
CHARTS[4].show(renderer="plotly_mimetype")
CHARTS[5].show(renderer="plotly_mimetype")

,fold,policy,rows,permitted,violating,query_policy
0,0,"no advertising: spam, referral links, unsolici...",629,379,250,True
1,0,no legal advice: do not offer or request legal...,1011,422,589,False
2,1,"no advertising: spam, referral links, unsolici...",849,474,375,False
3,1,no legal advice: do not offer or request legal...,366,148,218,True


## 6. Ablations and probability diagnostics

Combined-minus-one-family contrasts measure each family's conditional contribution. Higher AUC may coexist with worse log loss. Per-policy-ranked pooled AUC is reported separately from mean per-policy AUC, and neither is a hidden competition result.

In [7]:
display(pd.DataFrame(result["pooled_metrics"]))
CHARTS[6].show(renderer="plotly_mimetype")
CHARTS[7].show(renderer="plotly_mimetype")

,variant,macro_auc,pooled_auc,ranked_pooled_auc
0,lexical_control,0.657508,0.641345,0.650247
1,add_behavior,0.679862,0.680777,0.681784
2,add_act_roles,0.686099,0.691055,0.690944
3,copy_lexical,0.684815,0.690370,0.690285
4,condition_lexical,0.694518,0.698787,0.698227
5,copy_behavior,0.685083,0.689758,0.689774
6,condition_behavior,0.683636,0.688308,0.688402
7,copy_both,0.683560,0.688946,0.688677
8,condition_both,0.692901,0.697225,0.696993


## 7. Fixed decision and next gate

Primary: `condition_lexical`. Require +0.003 macro AUC, positive simultaneous lower bounds, and no policy regression against BOTH the actor anchor and `copy_lexical`; ranked pooled AUC cannot decline versus the actor anchor. Passing means eligibility for further validation only. The full interaction candidate cannot silently replace the primary.

Research basis: [Daumé III, feature augmentation](https://aclanthology.org/P07-1033/) and [policy-sensitive norm detection](https://aclanthology.org/2021.findings-emnlp.288/). No external datasets are added. [Rule By Example](https://aclanthology.org/2023.acl-long.22/) motivates a separate encoder-learning direction, not a claim about this sparse CPU study.

In [8]:
for requirement in result["primary_requirements"]:
    display(pd.DataFrame([requirement["contrast"]]))
    print("Reference:", requirement["reference"], "| Per-policy deltas:", requirement["per_policy_delta"])
print("Decision:", result["decision"])
for limitation in result["limitations"]:
    print(limitation)
print("Interactive dashboard:", write_dashboard(ROOT, result))
print("Private checkpoint directory:", ROOT / "runs/policy_features" / result["run_id"])

,comparison,candidate,reference,delta_auc,simultaneous_low,simultaneous_high,valid_draws
0,condition_lexical vs actor anchor,condition_lexical,add_act_roles,0.008419,-0.015461,0.0323,500


Reference: add_act_roles | Per-policy deltas: [0.010820895522388074, 0.006017494765268694]


,comparison,candidate,reference,delta_auc,simultaneous_low,simultaneous_high,valid_draws
0,conditioning: lexical,condition_lexical,copy_lexical,0.009704,-0.014177,0.033584,500


Reference: copy_lexical | Per-policy deltas: [0.013507462686567218, 0.005900080233263605]
Decision: DO_NOT_PROMOTE_PRIMARY
Exploratory follow-up chosen after Round 2, not a fresh holdout.
Actor anchor was not promoted; its reuse is diagnostic only.
The query rule has legal supplied labels in training: not zero-shot transfer.
Unknown policies receive no learned extra rule block; the shared features remain.
Intervals cover this round's 13 contrasts, not all adaptive project choices.
No new Kaggle score or accepted-Qwen-model comparison occurs in this round.
Coefficient plots show fitted associations, not causal feature value.


Interactive dashboard: /home/sagemaker-user/projects/jigsaw-rule-classifier/reports/policy_features/dashboard.html
Private checkpoint directory: /home/sagemaker-user/projects/jigsaw-rule-classifier/runs/policy_features/5927fe5eb15c3cf2d8aa
